# Laya V5.2 — GOLD-Aware Supervised Correction of a Compressed ModernBERT Model

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V52_Gold_Aware_Supervised_Colab.ipynb)

## The V5.1 result is **not 100% gold accuracy**

Your uploaded V5.1 result files show 120/120 agreement with the **V4.6 teacher** on the reused development PROBE, but both teacher and student answered **105/120 gold labels correctly (87.5%)**. Training stopped at *step 0*, before an optimizer update. The exact-match stop is functioning correctly for imitation, but cannot fix a teacher's mistakes.

V5.2 changes the objective from pure **teacher KL imitation** to **gold-label supervised learning**, with a small teacher-consistency regularizer. Supervised updates can intentionally disagree with an incorrect teacher answer.

The existing **20 compressed attention/MLP projection specifications** and the full 28-layer ModernBERT are retained, with ~130.60 MB predicted FP32 parameter savings. We reuse the deterministic V5.0 factor reconstruction, **not** the 520-trial brute-force sweep. By default, update only the lightweight scorer and question-type embeddings; optionally update compressed A/B factors to extend capacity (costs more GPU memory).

## Experimental protocol

- TRAIN: first 500 case-disjoint cases from the dataset train split (same seed as V5.1).
- HISTORICAL PROBE: the 120 historically used train-split cases; diagnostics only, **not checkpoint selection**.
- NEW GOLD VALIDATION: the next 120 train-split cases, unused for checkpoint selection in V5.1
  (but previously included as a V5.0 audit; they are NOT a pristine cross-study holdout); select checkpoints by **gold accuracy**, not teacher agreement. Reusing it for selection means it is no longer an independent final test.
- FINAL AUDIT: the distinct `LocalLLaMA/typed-decisions/all/test` split, evaluated **one time after the model is frozen**.
- Benchmark against V4.6; export a strict full FP32 standalone model; publish only if the *fresh test* beats the teacher and every other gate passes.

The soft gold probability labels are also available, but explicit gold class labels remain the principal objective. A 100% match with the teacher is **not** a stop condition: early stop triggers only after a predefined **gold-accuracy improvement** on the validation subset set aside from V5.1 selection.

The private-by-default HF destination is `vtava/Laya-v52-gold-aware-standalone`. **This model is experimental: neither improved gold accuracy nor a publication pass is guaranteed until its Colab run completes.**

[Original V5.1 Colab](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V51_All28_EarlyStop_Publish_Colab.ipynb) · [Standalone inference runtime](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/standalone/laya_v50_all28/modeling_laya_v50.py)


### Download all outputs

The final cell automatically creates `Laya_V52_All_Results.zip` and downloads it from Colab, **even if the Hugging Face publication gates fail**. The archive includes the complete exported standalone model, all training/probe/test reports, checkpoints, run summary, HF upload status and a per-file SHA-256 manifest. The model can make the ZIP larger than 1 GB; the notebook also displays a manual download link in case the browser blocks the automatic download.


In [1]:
#@title 1. Pinned requirements, GPU environment and runtime imports
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
import json,math,time,random,hashlib,copy,statistics,gc,importlib.util
from pathlib import Path
from collections import OrderedDict,Counter
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import HfApi,snapshot_download,notebook_login
from safetensors.torch import save_file,load_file
from transformers import AutoTokenizer
assert torch.cuda.is_available(),"Select a Colab GPU (T4 or better)"
DEVICE=torch.device("cuda")
print("Torch:",torch.__version__,"GPU:",torch.cuda.get_device_name(0))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 38.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
Torch: 2.11.0+cu130 GPU: Tesla T4


In [2]:
#@title 2. Gold-aware configuration, independent teacher and model source
TEACHER_REPO="vtava/Laya-1.2-speedup-standalone"
PINNED_TEACHER_REVISION="b60184b2976e9f7ffa60c1c42b221d81b3691c3d"
STUDENT_REPO="vtava/Laya-v52-gold-aware-standalone"
SEED=2047
MAX_LEN=512
TRAIN_CASES=500
DEV_CASES=240
BATCH=2 #@param {type:"integer"}
TRAIN_B=2 #@param {type:"integer"}
TRAIN_MAX_TOKENS=224 #@param {type:"integer"}
MAX_TRAIN_STEPS=400 #@param {type:"integer"}
CHECK_EVERY_STEPS=20 #@param {type:"integer"}
GOLD_SCORER_LR=0.000025 #@param {type:"number"}
GOLD_ADAPTER_LR=0.000003 #@param {type:"number"}
TRAIN_COMPRESSED_ADAPTERS=False #@param {type:"boolean"}
GOLD_CE_WEIGHT=0.75
SOFT_GOLD_WEIGHT=0.15
TEACHER_KL_WEIGHT=0.10
LABEL_SMOOTHING=0.02
TEMPERATURE=2.0
HISTORICAL_PROBE_DECISIONS=120
GOLD_VAL_DECISIONS=180
AUDIT_DECISIONS=200
BENCH_REPEATS=10 #@param {type:"integer"}
MIN_VALID_GOLD_GAIN_PP=1.0 #@param {type:"number"}
MIN_TEST_GOLD_GAIN_PP=1.0 #@param {type:"number"}
EARLY_STOP_GOLD_GAIN_PP=5.0 #@param {type:"number"}
MIN_VAL_TEACHER_AGREEMENT=0.82
MIN_TEST_TEACHER_AGREEMENT=0.80
MAX_TEST_MEAN_JS=0.06
MIN_SAVED_BYTES=20_000_000
MIN_GPU_SPEED_RATIO=0.95
MIN_WALL_SPEED_RATIO=0.95
REQUIRE_SPEED_NONREGRESSION=False #@param {type:"boolean"}
PUBLISH_TO_HF=True #@param {type:"boolean"}
PRIVATE_REPO=True #@param {type:"boolean"}
ALLOW_UPDATE_EXISTING=False #@param {type:"boolean"}
assert 1<=TRAIN_B<=4 and 1<=BATCH<=8
assert MAX_TRAIN_STEPS>=0 and CHECK_EVERY_STEPS>=1
assert TRAIN_MAX_TOKENS>=192
assert GOLD_CE_WEIGHT>0 and abs(GOLD_CE_WEIGHT+SOFT_GOLD_WEIGHT+TEACHER_KL_WEIGHT-1)<1e-8
assert 0<=LABEL_SMOOTHING<0.15
assert 0<MIN_VALID_GOLD_GAIN_PP<=EARLY_STOP_GOLD_GAIN_PP
DEVICE=torch.device("cuda")
device=DEVICE
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
OUT=Path("/content/Laya_V52_GoldAware")
OUT.mkdir(exist_ok=True,parents=True)
DEST=OUT/"standalone_model"
DEST.mkdir(exist_ok=True,parents=True)
api=HfApi()
try:
    teacher_info=api.model_info(TEACHER_REPO,revision=PINNED_TEACHER_REVISION)
except Exception:
    notebook_login()
    teacher_info=api.model_info(TEACHER_REPO,revision=PINNED_TEACHER_REVISION)
assert teacher_info.sha==PINNED_TEACHER_REVISION
parent_dir=Path(snapshot_download(TEACHER_REPO,revision=PINNED_TEACHER_REVISION))
source_config=json.loads((parent_dir/"config.json").read_text())
assert source_config["format"]=="laya-1p2-full-standalone-v1"
assert int(source_config["head_context_budget"])==64
source_cfg={"head_max_len":192,"max_len":MAX_LEN}
tokenizer=AutoTokenizer.from_pretrained(str(parent_dir/"tokenizer"),local_files_only=True)
assert tokenizer.pad_token_id is not None
print("Frozen original teacher:",teacher_info.sha)
print("Original model.safetensors bytes:",(parent_dir/"model.safetensors").stat().st_size)


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

modeling_laya_standalone.cpython-313.pyc:   0%|          | 0.00/21.0k [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

SOURCE_REVISIONS.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

...

size_report.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Frozen original teacher: b60184b2976e9f7ffa60c1c42b221d81b3691c3d
Original model.safetensors bytes: 1684141004


In [3]:
#@title 3. Exact standalone V5.0 runtime and frozen teacher/student
V50_RUNTIME_CODE = "\"\"\"Laya V5.0 all-28-layer brute-force hybrid SVD compression, fully standalone.\n\nRequires only torch, transformers, safetensors. Does NOT download or import\nconvaiinnovations/laya-typed-decisions or TinyCeNN-LM at inference.\nThe model weights and tokenizer must be present in the same export directory.\nOption-score model only; auxiliary action logits are intentionally unavailable.\n\"\"\"\nfrom __future__ import annotations\nimport json\nfrom pathlib import Path\nfrom typing import Any, Mapping, Sequence\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\nfrom safetensors.torch import load_file\nfrom transformers import AutoModel, AutoConfig, AutoTokenizer\n\nQTYPES = {\"choice\": 0, \"score\": 1, \"noul\": 2}\n_DEFAULT_NOUL_LABELS = {\"false\": \"false\", \"true\": \"true\"}\n\n\ndef _serialize_state(state: str | dict | list) -> str:\n    if isinstance(state, str):\n        return state\n    return json.dumps(state, ensure_ascii=False, separators=(\",\", \":\"), default=str)\n\n\ndef _render_value(value: Any) -> str:\n    if isinstance(value, str):\n        return value\n    return json.dumps(value, ensure_ascii=False, separators=(\", \", \": \"), default=str)\n\n\ndef normalize_question(question: Mapping[str, Any]) -> dict[str, Any]:\n    \"\"\"Normalize an external typed-decision question into the runtime format.\"\"\"\n    qtype = str(question.get(\"type\", question.get(\"t\", \"choice\"))).lower()\n    if qtype not in QTYPES:\n        raise ValueError(f\"unsupported question type: {qtype!r}\")\n\n    criteria = question.get(\"criteria\", question.get(\"crit\"))\n    if qtype == \"choice\":\n        if isinstance(criteria, (list, tuple)):\n            criteria = {str(x): None for x in criteria}\n        if not isinstance(criteria, Mapping) or len(criteria) < 2:\n            raise ValueError(\"choice questions require at least two criteria/options\")\n        criteria = dict(criteria)\n    elif qtype == \"score\":\n        if not isinstance(criteria, (list, tuple)) or len(criteria) < 2:\n            raise ValueError(\"score questions require at least two ordered criteria\")\n        criteria = list(criteria)\n    else:\n        if criteria is None:\n            criteria = {}\n        if not isinstance(criteria, Mapping):\n            raise ValueError(\"noul criteria must be a mapping when provided\")\n        criteria = {str(k).lower(): v for k, v in criteria.items()}\n\n    instructions = question.get(\"instructions\", question.get(\"ins\", \"\"))\n    if not isinstance(instructions, str):\n        instructions = json.dumps(instructions, ensure_ascii=False, default=str)\n\n    result = {\"t\": qtype, \"ins\": instructions, \"crit\": criteria}\n    if \"labels\" in question:\n        result[\"labels\"] = dict(question[\"labels\"])\n    return result\n\n\ndef render_options(question: Mapping[str, Any]) -> list[str]:\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    qtype, criteria = q[\"t\"], q.get(\"crit\")\n    if qtype == \"choice\":\n        return [\n            str(k) if v is None or v == \"\" else f\"{k}: {_render_value(v)}\"\n            for k, v in criteria.items()\n        ]\n    if qtype == \"score\":\n        return [f\"level {i}: {_render_value(v)}\" for i, v in enumerate(criteria)]\n\n    labels = q.get(\"labels\", _DEFAULT_NOUL_LABELS)\n    false_label = str(labels.get(\"false\", \"false\")).strip() or \"false\"\n    true_label = str(labels.get(\"true\", \"true\")).strip() or \"true\"\n    false_desc = criteria.get(\"false\")\n    true_desc = criteria.get(\"true\")\n    return [\n        f\"{false_label}: \" + (\n            _render_value(false_desc) if false_desc not in (None, \"\") else \"no, the statement does not hold\"\n        ),\n        f\"{true_label}: \" + (\n            _render_value(true_desc) if true_desc not in (None, \"\") else \"yes, the statement holds\"\n        ),\n    ]\n\n\ndef build_sequence(\n    tokenizer,\n    state: str | dict | list,\n    question: Mapping[str, Any],\n    *,\n    max_len: int = 512,\n    head_max_len: int = 192,\n    truncate_left: bool = False,\n    state_ids: Sequence[int] | None = None,\n) -> tuple[list[int], list[int]]:\n    \"\"\"Build a compact typed-decision sequence and option marker positions.\"\"\"\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    options = render_options(q)\n    mask_token = tokenizer.mask_token\n    instruction = str(q[\"ins\"]).replace(mask_token, \" \")\n    head_ids = tokenizer(\n        f\"{q['t']} question: {instruction}\", add_special_tokens=False\n    )[\"input_ids\"]\n\n    option_ids: list[list[int]] = []\n    for option in options:\n        ids = tokenizer(\n            \" \" + option.replace(mask_token, \" \"),\n            add_special_tokens=False,\n            truncation=True,\n            max_length=48,\n        )[\"input_ids\"]\n        option_ids.append([tokenizer.mask_token_id] + ids)\n\n    option_budget = head_max_len - sum(len(x) for x in option_ids)\n    if option_budget < 16:\n        per = max(4, (head_max_len - 16) // max(1, len(option_ids)))\n        option_ids = [x[:per] for x in option_ids]\n        option_budget = head_max_len - sum(len(x) for x in option_ids)\n    head_ids = head_ids[: max(8, option_budget)]\n\n    ids = [tokenizer.cls_token_id] + head_ids + [tokenizer.sep_token_id]\n    markers: list[int] = []\n    for option in option_ids:\n        markers.append(len(ids))\n        ids.extend(option)\n    ids.append(tokenizer.sep_token_id)\n\n    if state_ids is None:\n        text = _serialize_state(state).replace(mask_token, \" \")\n        state_ids = tokenizer(text, add_special_tokens=False)[\"input_ids\"]\n    room = max(0, max_len - len(ids) - 1)\n    if truncate_left:\n        start = max(0, len(state_ids) - room)\n        state_part = list(state_ids[start:])\n    else:\n        state_part = list(state_ids[:room])\n    ids = (ids + state_part + [tokenizer.sep_token_id])[:max_len]\n    markers = [m for m in markers if m < max_len]\n    return ids, markers\n\n\ndef collate_items(items: Sequence[Mapping[str, Any]], pad_id: int) -> dict[str, Tensor]:\n    if not items:\n        raise ValueError(\"items must not be empty\")\n    n = len(items)\n    length = max(len(x[\"ids\"]) for x in items)\n    kmax = max(len(x[\"markers\"]) for x in items)\n    ids = torch.full((n, length), int(pad_id), dtype=torch.long)\n    mask = torch.zeros((n, length), dtype=torch.long)\n    marker_pos = torch.zeros((n, kmax), dtype=torch.long)\n    marker_mask = torch.zeros((n, kmax), dtype=torch.bool)\n    qtype = torch.zeros(n, dtype=torch.long)\n\n    for i, item in enumerate(items):\n        row = torch.tensor(item[\"ids\"], dtype=torch.long)\n        ids[i, : row.numel()] = row\n        mask[i, : row.numel()] = 1\n        m = torch.tensor(item[\"markers\"], dtype=torch.long)\n        marker_pos[i, : m.numel()] = m\n        marker_mask[i, : m.numel()] = True\n        qtype[i] = int(item[\"qtype\"])\n    return {\n        \"input_ids\": ids,\n        \"attention_mask\": mask,\n        \"marker_pos\": marker_pos,\n        \"marker_mask\": marker_mask,\n        \"qtype\": qtype,\n    }\n\n\n\ndef _apply_rope_compat(cfg):\n    rope = getattr(cfg, \"rope_parameters\", None)\n    if not isinstance(rope,dict):\n        return\n    flat = rope.get(\"rope_theta\")\n    for layer_type, attr in ((\"full_attention\",\"global_rope_theta\"),\n                             (\"sliding_attention\",\"local_rope_theta\")):\n        v = rope.get(layer_type)\n        theta = v.get(\"rope_theta\") if isinstance(v,dict) else flat\n        if theta is not None and hasattr(cfg,attr):\n            setattr(cfg,attr,float(theta))\n\n\nclass LayaStandalone(nn.Module):\n    \"\"\"Native pretrained ModernBERT + exact 64-token option-scoring head.\n\n    ALL weights are stored locally in model.safetensors. No auxiliary act_head.\n    \"\"\"\n    def __init__(self, encoder_config=None, *, budget=64, head_layers=2,\n                 pretrained_components=None):\n        super().__init__()\n        self.budget = int(budget)\n        assert self.budget == 64, \"This checkpoint implements the validated 64-token head\"\n        if pretrained_components is not None:\n            # EXPORT ONLY: reuse the loaded upstream model without duplicating\n            # ~400M weights; this branch is not needed during standalone load.\n            self.encoder = pretrained_components.encoder\n            self.head = pretrained_components.head\n            self.type_emb = pretrained_components.type_emb\n            self.scorer = pretrained_components.scorer\n        else:\n            assert encoder_config is not None\n            _apply_rope_compat(encoder_config)\n            self.encoder = AutoModel.from_config(\n                encoder_config, attn_implementation=\"sdpa\")\n            d = int(encoder_config.hidden_size)\n            nhead = max(1,d//64)\n            layer = nn.TransformerEncoderLayer(\n                d,nhead,dim_feedforward=4*d,dropout=0.1,\n                batch_first=True,norm_first=True)\n            self.head = nn.TransformerEncoder(\n                layer,int(head_layers),enable_nested_tensor=False)\n            self.type_emb = nn.Embedding(3,d)\n            self.scorer = nn.Sequential(nn.LayerNorm(d),nn.Linear(d,d),\n                                        nn.GELU(),nn.Linear(d,1))\n\n    def _select_head_tokens(self,h,attention_mask,marker_pos,marker_mask):\n        # Copied from V4.6, keeping exact order/masks for reproduction.\n        b,t,d=h.shape\n        cls=torch.zeros((b,1),dtype=torch.long,device=h.device)\n        lm_count=min(self.budget,max(t-1,1))\n        lm=torch.linspace(1,max(t-1,1),steps=lm_count,device=h.device).round().long()\n        lm=lm.clamp(max=t-1).view(1,-1).expand(b,-1)\n        marker_ok=marker_mask.bool()\n        lm_ok=attention_mask.bool().gather(1,lm)\n        lm_ok=lm_ok & (lm!=0)\n        duplicate=(lm[:,:,None]==marker_pos[:,None,:]) & marker_ok[:,None,:]\n        lm_ok=lm_ok & ~duplicate.any(-1)\n        idx=torch.cat((cls,marker_pos.clamp(0,t-1),lm),dim=1)\n        keep=torch.cat((torch.ones((b,1),dtype=torch.bool,device=h.device),\n                        marker_ok,lm_ok),dim=1)\n        gathered=h.gather(1,idx[:,:,None].expand(-1,-1,d))\n        return gathered,keep\n\n    def forward(self,input_ids,attention_mask,marker_pos,marker_mask,qtype):\n        h=self.encoder(input_ids=input_ids,\n                       attention_mask=attention_mask).last_hidden_state\n        h=h+self.type_emb(qtype)[:,None,:]\n        compact,keep=self._select_head_tokens(\n            h,attention_mask,marker_pos,marker_mask)\n        transformed=self.head(compact,src_key_padding_mask=~keep)\n        marker_hidden=transformed[:,1:1+marker_pos.shape[1],:]\n        logits=self.scorer(marker_hidden).squeeze(-1).float()\n        logits=logits.masked_fill(~marker_mask,-1e4)\n        return logits,None\n\n    @torch.inference_mode()\n    def predict(self,tokenizer,state,questions,*,max_len=512,head_max_len=192):\n        items=[];names=[];labels=[]\n        for name,raw in questions.items():\n            q=normalize_question(raw)\n            if q[\"t\"]!=\"choice\":\n                raise ValueError(\"Only choice questions supported by predict()\")\n            opts=list(q[\"crit\"].keys())\n            ids,markers=build_sequence(tokenizer,state,q,max_len=max_len,\n                                       head_max_len=head_max_len)\n            if len(markers)!=len(opts):\n                raise ValueError(\"Truncated an option marker\")\n            names.append(name);labels.append(opts)\n            items.append(dict(ids=ids,markers=markers,qtype=QTYPES[q[\"t\"]]))\n        batch=collate_items(items,int(tokenizer.pad_token_id))\n        device=next(self.parameters()).device\n        batch={k:v.to(device) for k,v in batch.items()}\n        with (torch.autocast(\"cuda\",dtype=torch.float16)\n              if device.type==\"cuda\" else __import__(\"contextlib\").nullcontext()):\n            logits,_=self(**batch)\n        probs=logits.float().masked_fill(~batch[\"marker_mask\"],-1e9).softmax(-1).cpu()\n        answers={}\n        for i,name in enumerate(names):\n            p=probs[i,:len(labels[i])]\n            best=int(p.argmax())\n            answers[name]=dict(choice=labels[i][best],\n                               confidence=float(p[best]),\n                               probabilities={label:float(pr) for label,pr in zip(labels[i],p)})\n        return answers\n\n\ndef load_model(model_dir,*,device=\"cpu\",local_files_only=True,compile_head=False):\n    \"\"\"Load entirely from the local standalone directory. No network needed.\"\"\"\n    root=Path(model_dir).resolve()\n    metadata=json.loads((root/\"config.json\").read_text())\n    assert metadata[\"format\"]==\"laya-1p2-full-standalone-v1\"\n    enc_cfg=AutoConfig.from_pretrained(\n        str(root/\"encoder\"),local_files_only=local_files_only)\n    model=LayaStandalone(enc_cfg,budget=int(metadata[\"head_context_budget\"]),\n                         head_layers=int(metadata[\"head_layers\"]))\n    state=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(state,strict=True)\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tok=AutoTokenizer.from_pretrained(\n        str(root/\"tokenizer\"),local_files_only=local_files_only)\n    return model,tok\n\ndef local_pair_indices(model):\n    \"\"\"Verify FULL / same-window LOCAL / LOCAL triples in ModernBERT.\n\n    transformers==4.57.6 uses attn.local_attention: (-1,-1) means FULL,\n    (64,64) is a 128-token sliding window. Newer builds can expose\n    config.layer_types with attn.sliding_window instead. Support both.\n    \"\"\"\n    layers=model.encoder.layers\n    cfg=model.encoder.config\n    n=len(layers)\n    layer_types=getattr(cfg,\"layer_types\",None)\n    if layer_types is not None and len(layer_types)!=n:\n        raise ValueError(\"ModernBERT layer_types count does not match layers\")\n\n    def mode_at(i):\n        a=layers[i].attn\n        old_window=getattr(a,\"local_attention\",None)\n        if old_window is not None:\n            window=tuple(old_window)\n            if window==(-1,-1):\n                return (\"full\",None)\n            if len(window)!=2 or min(window)<0:\n                raise ValueError(f\"Malformed local attention window at {i}: {window}\")\n            return (\"local\",window)\n        typ=(layer_types[i] if layer_types is not None else\n             getattr(layers[i],\"attention_type\",None))\n        window=getattr(a,\"sliding_window\",None)\n        if typ==\"full_attention\" and window is None:\n            return (\"full\",None)\n        if typ==\"sliding_attention\" and window is not None:\n            return (\"local\",int(window))\n        raise ValueError(f\"Unsupported ModernBERT attention metadata at layer {i}: {typ}, {window}\")\n\n    pairs=[]\n    for anchor in range(0,n,3):\n        mode,_=mode_at(anchor)\n        if mode!=\"full\":\n            raise ValueError(f\"Layer {anchor} must be FULL attention\")\n        left_idx,right_idx=anchor+1,anchor+2\n        if right_idx>=n:\n            continue\n        m1,w1=mode_at(left_idx)\n        m2,w2=mode_at(right_idx)\n        if m1!=\"local\" or m2!=\"local\" or w1!=w2:\n            raise ValueError(\n                f\"Layers {(left_idx,right_idx)} must have identical local windows: {m1}/{w1}, {m2}/{w2}\")\n        pairs.append((left_idx,right_idx))\n    return pairs\n\n\nclass SVDDeltaLinear(nn.Module):\n    \"\"\"Second-layer linear projection = shared first-layer projection + A @ B.\n\n    The shared linear is a plain Python reference, NOT a registered submodule;\n    its weights appear only once in state_dict, under the original first layer.\n    \"\"\"\n    def __init__(self, first_linear, rank, *, A=None, B=None):\n        super().__init__()\n        if not isinstance(first_linear,nn.Linear):\n            raise TypeError(f\"Expected nn.Linear, got {type(first_linear)}\")\n        out_dim,in_dim=first_linear.weight.shape\n        rank=int(rank)\n        if not (1<=rank<min(in_dim,out_dim)):\n            raise ValueError(f\"Unsupported rank {rank} for {out_dim}x{in_dim}\")\n        # Using object.__setattr__ prevents double registering the shared module.\n        object.__setattr__(self,\"_shared\",first_linear)\n        self.in_features=int(in_dim)\n        self.out_features=int(out_dim)\n        self.max_rank=rank\n        self.active_rank=rank\n        self.A=nn.Parameter(torch.zeros((out_dim,rank),device=first_linear.weight.device,\n                                        dtype=first_linear.weight.dtype))\n        self.B=nn.Parameter(torch.zeros((rank,in_dim),device=first_linear.weight.device,\n                                        dtype=first_linear.weight.dtype))\n        if A is not None and B is not None:\n            with torch.no_grad():\n                self.A.copy_(A.to(self.A.device,self.A.dtype))\n                self.B.copy_(B.to(self.B.device,self.B.dtype))\n\n    def set_rank(self, rank):\n        rank=int(rank)\n        if rank<1 or rank>self.max_rank:\n            raise ValueError(f\"rank {rank} outside 1..{self.max_rank}\")\n        self.active_rank=rank\n\n    def shrink(self, rank):\n        \"\"\"Permanently discard extra SVD columns for a compact checkpoint.\"\"\"\n        rank=int(rank)\n        if rank<1 or rank>self.max_rank:\n            raise ValueError(rank)\n        self.A=nn.Parameter(self.A[:,:rank].detach().clone())\n        self.B=nn.Parameter(self.B[:rank,:].detach().clone())\n        self.max_rank=rank\n        self.active_rank=rank\n\n    def forward(self,x):\n        # fp16 autocast handles both paths identically, with FP32 stored weights.\n        w=self._shared.weight\n        bias=self._shared.bias\n        base=F.linear(x,w,bias)\n        r=self.active_rank\n        return base+F.linear(F.linear(x,self.B[:r,:]),self.A[:,:r])\n\n\n\n# V4.9 does NOT force compression of all nine local pairs.\n# Each selected follower projection gets a separate low-rank residual;\n# all unselected attention projection matrices retain pretrained weights.\ndef projection_slots(model):\n    slots=[]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            a=getattr(model.encoder.layers[first].attn,component)\n            b=getattr(model.encoder.layers[second].attn,component)\n            if not isinstance(a,nn.Linear) or not isinstance(b,nn.Linear):\n                raise TypeError(f\"Expected nn.Linear for layer {second}.{component}\")\n            if a.weight.shape != b.weight.shape:\n                raise ValueError(f\"Incompatible original projection: {second}.{component}\")\n            if a.bias is not None or b.bias is not None:\n                raise ValueError(f\"Biasful projection needs separate residual bias: {second}.{component}\")\n            slots.append((first,second,component))\n    return slots\n\n\ndef _validate_slot(model, first, second, component):\n    \"\"\"Validate schedule independently of other already-compressed slots.\"\"\"\n    first,second=int(first),int(second)\n    component=str(component)\n    if (first,second) not in local_pair_indices(model):\n        raise ValueError(f\"Unknown local pair: {(first,second)}\")\n    if component not in (\"Wqkv\",\"Wo\"):\n        raise ValueError(f\"Unknown projection: {component}\")\n    return first,second,component\n\n\ndef svd_factors(model, first, second, component, max_rank=256,\n                oversampling=8, niter=2):\n    \"\"\"Compute one factorization per projection from ORIGINAL pretrained weights.\n\n    A and B are float32 CPU tensors and can be sliced for different ranks.\n    \"\"\"\n    first,second,component=_validate_slot(model,first,second,component)\n    a=getattr(model.encoder.layers[first].attn,component)\n    b=getattr(model.encoder.layers[second].attn,component)\n    if not isinstance(a,nn.Linear) or not isinstance(b,nn.Linear):\n        raise ValueError(\"SVD must be done BEFORE that follower is compressed\")\n    with torch.no_grad():\n        delta=b.weight.detach().float()-a.weight.detach().float()\n        r=min(int(max_rank),min(delta.shape)-1)\n        q=min(r+int(oversampling),min(delta.shape))\n        # Randomized SVD (approximation, not an exact mathematical SVD).\n        U,S,V=torch.svd_lowrank(delta,q=q,niter=int(niter))\n        root=S[:r].clamp_min(0).sqrt()\n        A=(U[:,:r]*root.unsqueeze(0)).cpu().contiguous()\n        B=(root.unsqueeze(1)*V[:,:r].T).cpu().contiguous()\n        total=float(delta.square().sum().item())\n        error=float((delta-(A.to(delta.device)@B.to(delta.device))).square().sum().item())\n        del U,S,V,delta\n    return A,B,{\"rank\":r,\n        \"relative_frobenius_error\":(error/max(total,1e-30))**0.5,\n        \"saved_parameter_bytes_if_rank256\":None}\n\n\ndef build_adapter(model, first, second, component, rank, factors=None,\n                  *, initialized=True):\n    \"\"\"Replace ONE follower's projection; return original module for rollback.\n\n    If initialized=False, A/B are zeros until strictly loaded from checkpoint.\n    \"\"\"\n    first=int(first);second=int(second);component=str(component)\n    if (first,second) not in local_pair_indices(model):\n        raise ValueError(f\"Invalid local pair: {(first,second)}\")\n    leader=getattr(model.encoder.layers[first].attn,component)\n    follower_block=model.encoder.layers[second].attn\n    old=getattr(follower_block,component)\n    if not isinstance(leader,nn.Linear) or not isinstance(old,nn.Linear):\n        raise TypeError(\"Projection already wrapped or unavailable\")\n    if leader.bias is not None or old.bias is not None:\n        raise ValueError(\"Biasful attention projections are unsupported\")\n    if initialized:\n        if factors is None:\n            A,B,_=svd_factors(model,first,second,component,max_rank=rank)\n        else:\n            A,B=factors\n        adapter=SVDDeltaLinear(leader,rank,A=A[:,:rank],B=B[:rank,:])\n    else:\n        adapter=SVDDeltaLinear(leader,rank)\n    setattr(follower_block,component,adapter)\n    return old\n\n\ndef restore_original_projection(model, second, component, original):\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Rollback needs the original pretrained linear module\")\n    setattr(model.encoder.layers[int(second)].attn,str(component),original)\n\n\ndef selected_adapters(model):\n    selected=[]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            layer=getattr(model.encoder.layers[second].attn,component)\n            if isinstance(layer,SVDDeltaLinear):\n                selected.append((first,second,component,layer))\n    return selected\n\n\ndef unique_parameter_count(model):\n    return sum(p.numel() for p in model.parameters())\n\n\ndef estimated_saved_bytes(model, first, second, component, rank):\n    leader=getattr(model.encoder.layers[first].attn,component)\n    outdim,indim=leader.weight.shape\n    # Full follower weight removed; A (out×rank), B (rank×in) remain.\n    return 4*(outdim*indim-int(rank)*(outdim+indim))\n\n\ndef load_v49_model(model_dir, *, device=\"cpu\", compile_head=False):\n    \"\"\"Strict, fully offline restoration of selectively compressed model.\"\"\"\n    root=Path(model_dir).resolve()\n    meta=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if meta[\"format\"]!=\"laya-v49-selective-svd-v1\":\n        raise ValueError(\"Wrong model format: expected V4.9 selective SVD\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(meta[\"head_context_budget\"]),\n                         head_layers=int(meta[\"head_layers\"]))\n    for item in meta[\"compressed_projections\"]:\n        build_adapter(model,int(item[\"first\"]),int(item[\"second\"]),\n                      str(item[\"component\"]),int(item[\"rank\"]),\n                      initialized=False)\n    weights=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(weights,strict=True)\n    # No duplicated original follower weights or duplicated leader submodule.\n    assert len(selected_adapters(model))==len(meta[\"compressed_projections\"])\n    for first,second,component,wrapper in selected_adapters(model):\n        leader=getattr(model.encoder.layers[first].attn,component)\n        assert wrapper._shared is leader\n        assert \"_shared\" not in wrapper._modules\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tokenizer=AutoTokenizer.from_pretrained(str(root/\"tokenizer\"),local_files_only=True)\n    return model,tokenizer\n\n\n# ---------------------------------------------------------------------------\n# V5.0: brute-force all 28 ModernBERT blocks, attention + FFN, preserving the\n# proven V4.9 pretrained-pair DELTA path as one selectable candidate family.\n# ---------------------------------------------------------------------------\n\nclass IndependentSVDLinear(nn.Module):\n    \"\"\"Plain original projection compressed as A @ B; original dense W omitted.\n\n    Compatible with ModernBERT's attention Wqkv, Wo and MLP Wi, Wo calls.\n    A saved bias is kept if the original linear was biased.\n    \"\"\"\n    def __init__(self, original: nn.Linear, rank: int, A=None, B=None):\n        super().__init__()\n        if not isinstance(original, nn.Linear):\n            raise TypeError(\"Expected original nn.Linear\")\n        out_dim,in_dim=original.weight.shape\n        rank=int(rank)\n        if rank<=0 or rank>=min(out_dim,in_dim):\n            raise ValueError(f\"Bad SVD rank {rank} for {out_dim}x{in_dim}\")\n        self.in_features=int(in_dim)\n        self.out_features=int(out_dim)\n        self.rank=rank\n        self.A=nn.Parameter(torch.zeros((out_dim,rank),\n            dtype=original.weight.dtype,device=original.weight.device))\n        self.B=nn.Parameter(torch.zeros((rank,in_dim),\n            dtype=original.weight.dtype,device=original.weight.device))\n        if original.bias is None:\n            self.register_parameter(\"bias\",None)\n        else:\n            self.bias=nn.Parameter(original.bias.detach().clone())\n        if A is not None and B is not None:\n            with torch.no_grad():\n                self.A.copy_(A[:,:rank].to(device=self.A.device,dtype=self.A.dtype))\n                self.B.copy_(B[:rank,:].to(device=self.B.device,dtype=self.B.dtype))\n\n    def forward(self,x):\n        return F.linear(F.linear(x,self.B),self.A,self.bias)\n\n\ndef all_encoder_slots(model):\n    \"\"\"Enumerate available projections in EVERY pretrained encoder layer.\"\"\"\n    slots=[]\n    for index,layer in enumerate(model.encoder.layers):\n        for block,names in ((\"attn\",(\"Wqkv\",\"Wo\")),(\"mlp\",(\"Wi\",\"Wo\"))):\n            component=getattr(layer,block)\n            for name in names:\n                module=getattr(component,name,None)\n                if module is None:\n                    raise ValueError(f\"Required projection layer {index}.{block}.{name} missing\")\n                if not isinstance(module,nn.Linear):\n                    raise TypeError(f\"Not an uncompressed nn.Linear: {index}.{block}.{name}: {type(module)}\")\n                slots.append((index,block,name))\n    if len(slots)!=4*len(model.encoder.layers):\n        raise RuntimeError(\"Expected exactly two attention and two FFN projections per layer\")\n    return slots\n\n\ndef list_candidate_families(model):\n    \"\"\"Full brute-force search space: 112 own-SVD + 18 delta-shared slots.\"\"\"\n    options=[{\"layer\":int(layer),\"block\":block,\"component\":name,\n              \"mode\":\"own_svd\"}\n             for layer,block,name in all_encoder_slots(model)]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            options.append({\"layer\":int(second),\"leader\":int(first),\n                            \"block\":\"attn\",\"component\":component,\n                            \"mode\":\"pair_delta\"})\n    return options\n\n\ndef own_svd_factors(original, *, max_rank=256, oversampling=8,niter=2):\n    \"\"\"Randomized, truncated SVD of ORIGINAL pretrained W (not layer delta).\"\"\"\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Expected an uncompressed linear projection\")\n    with torch.no_grad():\n        weight=original.weight.detach().float()\n        rank=min(int(max_rank),min(weight.shape)-1)\n        q=min(rank+int(oversampling),min(weight.shape))\n        U,S,V=torch.svd_lowrank(weight,q=q,niter=int(niter))\n        scale=S[:rank].clamp_min(0).sqrt()\n        A=(U[:,:rank]*scale[None,:]).cpu().contiguous()\n        B=(scale[:,None]*V[:,:rank].T).cpu().contiguous()\n        approximated=A.to(weight.device)@B.to(weight.device)\n        numerator=float((weight-approximated).square().sum().item())\n        denominator=float(weight.square().sum().item())\n    return A,B,{\"rank\":rank,\n                \"relative_frobenius_error\":(numerator/max(denominator,1e-30))**.5}\n\n\ndef projection_at(model, spec):\n    return getattr(getattr(model.encoder.layers[int(spec[\"layer\"])],\n                           str(spec[\"block\"])),str(spec[\"component\"]))\n\n\ndef replace_projection(model, spec, *, factors=None, initialized=True):\n    \"\"\"Apply one projection replacement; return original module for rollback.\"\"\"\n    idx=int(spec[\"layer\"])\n    block=str(spec[\"block\"])\n    comp=str(spec[\"component\"])\n    rank=int(spec[\"rank\"])\n    mode=str(spec[\"mode\"])\n    owner=getattr(model.encoder.layers[idx],block)\n    old=getattr(owner,comp)\n    if not isinstance(old,nn.Linear):\n        raise TypeError(f\"Already compressed or unsupported {idx}.{block}.{comp}\")\n    if mode==\"own_svd\":\n        if initialized and factors is None:\n            factors=own_svd_factors(old,max_rank=rank)[:2]\n        if initialized:\n            A,B=factors\n            replacement=IndependentSVDLinear(old,rank,A,B)\n        else:\n            replacement=IndependentSVDLinear(old,rank)\n    elif mode==\"pair_delta\":\n        if block!=\"attn\" or comp not in (\"Wqkv\",\"Wo\"):\n            raise ValueError(\"Pair delta only supported for local attention\")\n        leader=int(spec[\"leader\"])\n        if (leader,idx) not in local_pair_indices(model):\n            raise ValueError(f\"Not a valid local pair: {leader},{idx}\")\n        shared=getattr(model.encoder.layers[leader].attn,comp)\n        if not isinstance(shared,nn.Linear):\n            raise TypeError(\"Leader must retain original pretrained linear\")\n        if shared.bias is not None or old.bias is not None:\n            raise ValueError(\"Pair delta only supported for bias-free attention\")\n        if initialized:\n            if factors is None:\n                A,B,_=svd_factors(model,leader,idx,comp,max_rank=rank)\n            else:\n                A,B=factors\n            replacement=SVDDeltaLinear(shared,rank,A=A[:,:rank],B=B[:rank,:])\n        else:\n            replacement=SVDDeltaLinear(shared,rank)\n    else:\n        raise ValueError(f\"Unsupported family: {mode}\")\n    setattr(owner,comp,replacement)\n    return old\n\n\ndef revert_projection(model,spec,original):\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Original projection must be nn.Linear\")\n    owner=getattr(model.encoder.layers[int(spec[\"layer\"])],str(spec[\"block\"]))\n    setattr(owner,str(spec[\"component\"]),original)\n\n\ndef applied_adapters(model,selected_specs):\n    result=[]\n    for spec in selected_specs:\n        wrapper=projection_at(model,spec)\n        if not isinstance(wrapper,(IndependentSVDLinear,SVDDeltaLinear)):\n            raise AssertionError(f\"Missing compressed adapter for {spec}\")\n        result.append(wrapper)\n    return result\n\n\ndef expected_saved_bytes(model,spec):\n    \"\"\"FP32 parameter saving for one replacement, including any old bias.\"\"\"\n    idx=int(spec[\"layer\"])\n    owner=getattr(model.encoder.layers[idx],str(spec[\"block\"]))\n    original=getattr(owner,str(spec[\"component\"]))\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Call before installing candidate\")\n    out_dim,in_dim=original.weight.shape\n    rank=int(spec[\"rank\"])\n    # Bias exists in both old and new for own_svd; cancels out.\n    # Pair delta has no bias.\n    return int(4*(out_dim*in_dim-rank*(out_dim+in_dim)))\n\n\ndef load_v50_model(model_dir, *, device=\"cpu\",compile_head=False):\n    \"\"\"Strict reload of all hybrid low-rank projections from local disk only.\"\"\"\n    root=Path(model_dir).resolve()\n    info=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if info[\"format\"]!=\"laya-v50-all28-bruteforce-v1\":\n        raise ValueError(\"Not a V5.0 hybrid model\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(info[\"head_context_budget\"]),\n                         head_layers=int(info[\"head_layers\"]))\n    # Rebuild in the same order: only uncompressed leaders are valid for delta.\n    for spec in info[\"compressed_projections\"]:\n        replace_projection(model,spec,initialized=False)\n    full=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(full,strict=True)\n    for spec,adapter in zip(info[\"compressed_projections\"],\n                            applied_adapters(model,info[\"compressed_projections\"])):\n        if spec[\"mode\"]==\"pair_delta\":\n            leader=getattr(model.encoder.layers[int(spec[\"leader\"])].attn,\n                           str(spec[\"component\"]))\n            assert adapter._shared is leader\n            assert \"_shared\" not in adapter._modules\n    model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tok=AutoTokenizer.from_pretrained(str(root/\"tokenizer\"),local_files_only=True)\n    return model,tok\n"
runtime_file=OUT/"modeling_laya_v50.py"
runtime_file.write_text(V50_RUNTIME_CODE,encoding="utf-8")
spec=importlib.util.spec_from_file_location("v51_runtime",str(runtime_file))
assert spec and spec.loader
v50=importlib.util.module_from_spec(spec)
spec.loader.exec_module(v50)
normalize_question=v50.normalize_question
build_sequence=v50.build_sequence
collate_items=v50.collate_items
QTYPES=v50.QTYPES
teacher,_=v50.load_model(parent_dir,device="cuda")
student,_=v50.load_model(parent_dir,device="cuda")
teacher.eval().requires_grad_(False)
student.eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
student.encoder.config.reference_compile=False
native_unique=v50.unique_parameter_count(teacher)
assert len(student.encoder.layers)==28
assert len(v50.all_encoder_slots(student))==112
print("Frozen V4.6 reference loaded; unique parameters:",f"{native_unique:,}")

# Regression test: V5.0/V5.1 pair-delta constructor MUST pass SVD factors by
# keyword. Run before expensive 20-projection setup, and restore source weight.
left,right=v50.local_pair_indices(student)[0]
smoke_spec=dict(layer=right,leader=left,block="attn",
                component="Wo",mode="pair_delta",rank=8)
old_linear=v50.projection_at(student,smoke_spec)
shared_linear=student.encoder.layers[left].attn.Wo
H,W=old_linear.weight.shape
test_A=torch.zeros((H,8),device=DEVICE,dtype=shared_linear.weight.dtype)
test_B=torch.zeros((8,W),device=DEVICE,dtype=shared_linear.weight.dtype)
original_ref=v50.replace_projection(
    student,smoke_spec,factors=(test_A,test_B),initialized=True)
try:
    adapter=v50.projection_at(student,smoke_spec)
    assert isinstance(adapter,v50.SVDDeltaLinear)
    assert adapter._shared is shared_linear
    assert "_shared" not in adapter._modules
    x=torch.randn(2,3,W,device=DEVICE,dtype=shared_linear.weight.dtype)
    with torch.no_grad():
        torch.testing.assert_close(adapter(x),shared_linear(x),atol=1e-5,rtol=1e-5)
finally:
    v50.revert_projection(student,smoke_spec,original_ref)
assert v50.projection_at(student,smoke_spec) is old_linear
del x,test_A,test_B,adapter,old_linear,shared_linear,original_ref
print("PASS: pair-delta keyword factors, forward pass and rollback")



Frozen V4.6 reference loaded; unique parameters: 421,029,889
PASS: pair-delta keyword factors, forward pass and rollback


In [4]:
#@title 4. CASE-DISJOINT TRAIN / PROBE / AUDIT benchmark from V4.6
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

# Preserve V5.0's exact case-level TRAIN and PROBE split to reproduce its
# 120/120 step-120 finding. Use the DATASET'S DISTINCT TEST SPLIT for AUDIT.
# Reuse historical V5.1 train/PROBE for comparability, but SELECT using
# unused second-half DEV cases. TEST split stays unseen until after freezing.
rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="train"))
random.Random(SEED).shuffle(rows)
assert TRAIN_CASES+DEV_CASES<=len(rows)
train_rows=rows[:TRAIN_CASES]
historical_probe_rows=rows[TRAIN_CASES:TRAIN_CASES+DEV_CASES//2]
gold_val_rows=rows[TRAIN_CASES+DEV_CASES//2:TRAIN_CASES+DEV_CASES]
train_items=make_items_from_rows(train_rows,SEED,True)
probe_items=make_items_from_rows(historical_probe_rows,SEED+1,True)
gold_val_items=make_items_from_rows(gold_val_rows,SEED+2,True)
# Case namespaces protect against overlapping local split ids.
test_rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="test"))
random.Random(SEED+9001).shuffle(test_rows)
audit_rows=[dict(row,id="TEST::"+str(row.get("id",i)))
            for i,row in enumerate(test_rows)]
audit_items=make_items_from_rows(audit_rows,SEED+9001,True)
assert len(probe_items)>=HISTORICAL_PROBE_DECISIONS
assert len(gold_val_items)>=GOLD_VAL_DECISIONS
assert len(audit_items)>=AUDIT_DECISIONS
groups=[{x["case_id"] for x in seq} for seq in
        (train_items,probe_items,gold_val_items,audit_items)]
for i in range(len(groups)):
    for j in range(i+1,len(groups)):
        assert groups[i].isdisjoint(groups[j]),(i,j)
def direct_batch(selected):
    b=collate_items(selected,int(tokenizer.pad_token_id))
    return {k:v.to(device) for k,v in b.items()}
def batch(items,offset,n=BATCH):
    return direct_batch([items[(offset+i)%len(items)] for i in range(n)])
print("TRAIN:",len(train_items),"OLD PROBE:",len(probe_items),
      "NEW GOLD VAL:",len(gold_val_items),"FINAL TEST:",len(audit_items))


README.md: 0.00B [00:00, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/599k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/222k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]

TRAIN: 2500 OLD PROBE: 600 NEW GOLD VAL: 600 FINAL TEST: 2000


In [5]:
#@title 5. Accuracy, divergence and paired CUDA/wall timing
@torch.inference_mode()
def probs_for(model,b):
    with torch.autocast("cuda",dtype=amp_dtype):
        scores,_=model(**b)
    return torch.softmax(scores.float().masked_fill(
        ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()

@torch.inference_mode()
def get_reference(items,limit):
    out=[]
    for start in range(0,min(limit,len(items)),BATCH):
        chosen=items[start:min(start+BATCH,limit)]
        p=probs_for(teacher,direct_batch(chosen))
        for j,item in enumerate(chosen):
            out.append(dict(probs=p[j],teacher=int(p[j].argmax()),
                gold=int(item["gold_index"]),case=item["case_id"],qid=item["qid"]))
    return out

@torch.inference_mode()
def quality(model,items,ref):
    agree=correct=tcorrect=0;js=0.
    n=len(ref)
    for start in range(0,n,BATCH):
        chosen=items[start:min(n,start+BATCH)]
        p=probs_for(model,direct_batch(chosen))
        for j,item in enumerate(chosen):
            t=ref[start+j]["probs"]; s=p[j]
            assert s.shape==t.shape
            m=(s+t)/2
            js+=float(.5*((s*(s.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()+
                            (t*(t.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()))
            guess=int(s.argmax())
            agree+=int(guess==ref[start+j]["teacher"])
            correct+=int(guess==item["gold_index"])
            tcorrect+=int(ref[start+j]["teacher"]==item["gold_index"])
    return dict(decisions=n,teacher_agreement=agree/n,gold_accuracy=correct/n,
        teacher_gold_accuracy=tcorrect/n,gold_drop=(tcorrect-correct)/n,mean_js=js/n)

BENCH_BATCHES=[batch(probe_items,0,BATCH),batch(probe_items,11,BATCH)]
@torch.inference_mode()
def benchmark_paired(reference,candidate,repeats=BENCH_REPEATS):
    """Alternating A/B/B/A with CUDA events AND synchronized wall-clock timings.
    All timings exclude Colab compilation, dataset loading and model downloading."""
    results=[]
    for j,b in enumerate(BENCH_BATCHES):
        def once(m):
            torch.cuda.synchronize()
            a=torch.cuda.Event(enable_timing=True)
            z=torch.cuda.Event(enable_timing=True)
            before=time.perf_counter()
            a.record()
            with torch.autocast("cuda",dtype=amp_dtype):
                outputs=m(**b)
            z.record();z.synchronize()
            wall=(time.perf_counter()-before)*1000
            return float(a.elapsed_time(z)),float(wall)
        for k in range(4):
            once(reference);once(candidate)
        d={"reference":[],"candidate":[]}
        w={"reference":[],"candidate":[]}
        for k in range(repeats):
            order=("reference","candidate","candidate","reference") if k%2==0 else (
                "candidate","reference","reference","candidate")
            for name in order:
                gpu,cpu=once(reference if name=="reference" else candidate)
                d[name].append(gpu);w[name].append(cpu)
        ref=float(np.median(d["reference"]))
        cand=float(np.median(d["candidate"]))
        rw=float(np.median(w["reference"]))
        cw=float(np.median(w["candidate"]))
        results.append(dict(shape=j,batch=int(b["input_ids"].shape[0]),
            tokens=int(b["input_ids"].shape[1]),
            native_gpu_ms=ref,candidate_gpu_ms=cand,gpu_speedup=ref/max(cand,1e-9),
            native_wall_ms=rw,candidate_wall_ms=cw,wall_speedup=rw/max(cw,1e-9),
            native_p95_ms=float(np.percentile(d["reference"],95)),
            candidate_p95_ms=float(np.percentile(d["candidate"],95)),n=len(d["reference"])))
    return dict(median_gpu_speedup=float(np.median([r["gpu_speedup"] for r in results])),
        min_gpu_speedup=float(min(r["gpu_speedup"] for r in results)),
        median_wall_speedup=float(np.median([r["wall_speedup"] for r in results])),
        min_wall_speedup=float(min(r["wall_speedup"] for r in results)),
        per_shape=results)

def valid_quality(q):
    return (q["teacher_agreement"]>=MIN_TEACHER_AGREEMENT and
            q["gold_drop"]<=MAX_GOLD_DROP_PP/100 and q["mean_js"]<=MAX_JS)
def valid_speed(s):
    return (s["min_gpu_speedup"]>=MIN_GPU_SPEED_RATIO and
            s["min_wall_speedup"]>=MIN_WALL_SPEED_RATIO)



In [6]:
#@title 6. Reconstruct ONLY the 20 proven V5.0 compressed projections
# This exact ordered list comes from the user's V5.0 greedy_combinations.json.
# It replaces the 520-candidate rank sweep; all unchanged projections are
# restored from PINNED pretrained V4.6 weights.
SELECTED_SPECS = [
  {
    "layer": 26,
    "leader": 25,
    "block": "attn",
    "component": "Wqkv",
    "mode": "pair_delta",
    "rank": 64
  },
  {
    "layer": 26,
    "leader": 25,
    "block": "attn",
    "component": "Wo",
    "mode": "pair_delta",
    "rank": 64
  },
  {
    "layer": 17,
    "leader": 16,
    "block": "attn",
    "component": "Wo",
    "mode": "pair_delta",
    "rank": 256
  },
  {
    "layer": 20,
    "leader": 19,
    "block": "attn",
    "component": "Wo",
    "mode": "pair_delta",
    "rank": 256
  },
  {
    "layer": 27,
    "block": "attn",
    "component": "Wqkv",
    "mode": "own_svd",
    "rank": 64
  },
  {
    "layer": 24,
    "block": "mlp",
    "component": "Wi",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 23,
    "block": "attn",
    "component": "Wqkv",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 27,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 32
  },
  {
    "layer": 23,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 32
  },
  {
    "layer": 23,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 25,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 24,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 64
  },
  {
    "layer": 26,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 21,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 21,
    "block": "mlp",
    "component": "Wi",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 20,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 8,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 3,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 2,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 15,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  }
]
assert len(SELECTED_SPECS)==20
assert len({(s["layer"],s["block"],s["component"]) for s in SELECTED_SPECS})==20
assert sum(s["mode"]=="pair_delta" for s in SELECTED_SPECS)==4
# Stable fingerprint of the exact 20 selected projections and ranks.
# Used in model config, provenance and release verification.
spec_hash=hashlib.sha256(
    json.dumps(SELECTED_SPECS,sort_keys=True,separators=(",",":")).encode("utf-8")
).hexdigest()
assert len(spec_hash)==64
print("Compression specification SHA-256:",spec_hash)
def family_key(f):
    return (int(f["layer"]),str(f["block"]),str(f["component"]),str(f["mode"]))
def deterministic_factors(f):
    family_seed=SEED+sum((i+1)*ord(ch) for i,ch in enumerate(
        ":".join(map(str,family_key(f)))))
    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
        torch.manual_seed(family_seed)
        if f["mode"]=="own_svd":
            return v50.own_svd_factors(
                v50.projection_at(student,f),
                max_rank=256,oversampling=8,niter=2)[:2]
        return v50.svd_factors(
            student,int(f["leader"]),int(f["layer"]),
            str(f["component"]),max_rank=256,oversampling=8,niter=2)[:2]

PROBE_REF=get_reference(probe_items,HISTORICAL_PROBE_DECISIONS)
GOLD_VAL_REF=get_reference(gold_val_items,GOLD_VAL_DECISIONS)

native_q=quality(student,probe_items,PROBE_REF)
assert native_q["teacher_agreement"]==1.0,native_q
for j,s in enumerate(SELECTED_SPECS,1):
    factors=deterministic_factors(s)
    old=v50.replace_projection(student,s,factors=factors,initialized=True)
    assert v50.projection_at(student,s) is not old
    print(f"Selected {j:02d}/{len(SELECTED_SPECS)}: layer={s['layer']} "
          f"{s['block']}.{s['component']} {s['mode']} rank={s['rank']}")
unique_student=v50.unique_parameter_count(student)
estimated_bytes=(native_unique-unique_student)*4
assert estimated_bytes>0
initial_probe=quality(student,probe_items,PROBE_REF)
initial_gold_validation=quality(student,gold_val_items,GOLD_VAL_REF)
print("Historical initial PROBE (not used for selection):",initial_probe)
print("New gold-validation baseline:",initial_gold_validation)
print("Teacher gold accuracy in gold-validation:",initial_gold_validation["teacher_gold_accuracy"])

print("Projected parameter savings:",f"{estimated_bytes:,} bytes",
      f"({estimated_bytes/1e6:.3f} MB)")
(OUT/"selected_projections.json").write_text(json.dumps({
    "source_repo":TEACHER_REPO,"source_revision":PINNED_TEACHER_REVISION,
    "specifications":SELECTED_SPECS,"estimated_saved_fp32_bytes":estimated_bytes,
    "initial_probe":initial_probe},indent=2),encoding="utf-8")


Compression specification SHA-256: 5cb1804e2e2bea0180c19f3179f3075c9be467e49cc3ac99257f3318f4c6ac6e
Selected 01/20: layer=26 attn.Wqkv pair_delta rank=64
Selected 02/20: layer=26 attn.Wo pair_delta rank=64
Selected 03/20: layer=17 attn.Wo pair_delta rank=256
Selected 04/20: layer=20 attn.Wo pair_delta rank=256
Selected 05/20: layer=27 attn.Wqkv own_svd rank=64
Selected 06/20: layer=24 mlp.Wi own_svd rank=256
Selected 07/20: layer=23 attn.Wqkv own_svd rank=256
Selected 08/20: layer=27 attn.Wo own_svd rank=32
Selected 09/20: layer=23 attn.Wo own_svd rank=32
Selected 10/20: layer=23 mlp.Wo own_svd rank=256
Selected 11/20: layer=25 mlp.Wo own_svd rank=256
Selected 12/20: layer=24 mlp.Wo own_svd rank=64
Selected 13/20: layer=26 mlp.Wo own_svd rank=256
Selected 14/20: layer=21 mlp.Wo own_svd rank=256
Selected 15/20: layer=21 mlp.Wi own_svd rank=256
Selected 16/20: layer=20 mlp.Wo own_svd rank=256
Selected 17/20: layer=8 attn.Wo own_svd rank=256
Selected 18/20: layer=3 attn.Wo own_svd rank=25

2932

In [7]:
#@title 7. Audit label consistency; train GOLD supervision (not teacher-only imitation)
# A useful model must correct some teacher mistakes. This stage updates the
# scorer and type embedding; optional selected A/B factors add more capacity.
student.requires_grad_(False)
for p in student.scorer.parameters(): p.requires_grad_(True)
for p in student.type_emb.parameters(): p.requires_grad_(True)
adapter_modules=v50.applied_adapters(student,SELECTED_SPECS)
if TRAIN_COMPRESSED_ADAPTERS:
    for module in adapter_modules:
        module.A.requires_grad_(True)
        module.B.requires_grad_(True)
groups=[
 dict(params=list(student.scorer.parameters())+
             list(student.type_emb.parameters()),lr=GOLD_SCORER_LR),
]
if TRAIN_COMPRESSED_ADAPTERS:
    groups.append(dict(params=[
        p for m in adapter_modules for p in (m.A,m.B)],lr=GOLD_ADAPTER_LR))
trainable=[p for p in student.parameters() if p.requires_grad]
assert trainable and len({id(p) for p in trainable})==len(trainable)
# Numeric self-test for padded options: masked-option label smoothing
# MUST have finite, reasonably scaled gradients.
with torch.enable_grad():
    sample_scores=torch.tensor([[2.,1.,-1e9],[0.,3.,-1e9]],
        dtype=torch.float32,device=DEVICE,requires_grad=True)
    sample_valid=torch.tensor([[True,True,False],[True,True,False]],
        device=DEVICE)
    sample_hard=torch.tensor([0,1],device=DEVICE)
    sample_logp=F.log_softmax(sample_scores,dim=-1)
    smooth_target=(1-LABEL_SMOOTHING)*F.one_hot(sample_hard,3).float()
    smooth_target+=LABEL_SMOOTHING*sample_valid.float()/sample_valid.sum(
        -1,keepdim=True).clamp_min(1)
    sample_ce=-(smooth_target*sample_logp).sum(-1).mean()
    sample_ce.backward()
    assert 0<sample_ce.item()<2,(sample_ce.item(),"invalid-label smoothing")
    assert bool(torch.isfinite(sample_scores.grad).all())
    del sample_scores,sample_valid,sample_hard,sample_logp,smooth_target,sample_ce
print("PASS: valid-option gold loss and gradient sanity check")

print("Trainable GOLD-aware parameters:",sum(p.numel() for p in trainable),
      "adapters enabled:",TRAIN_COMPRESSED_ADAPTERS)
student.eval() # Dropout OFF, gradients ON.
BEST_FILE=OUT/"best_gold_trainable.safetensors"
BEST_META=OUT/"best_gold_trainable.json"
TRAIN_LOG=OUT/"gold_training_progress.json"
TRAINABLE_PARAMETER_NAMES=[
    name for name,p in student.named_parameters() if p.requires_grad]
parameter_names_and_shapes={name:list(p.shape)
                             for name,p in student.named_parameters() if p.requires_grad}
training_fingerprint=hashlib.sha256(json.dumps(
    parameter_names_and_shapes,sort_keys=True).encode()).hexdigest()

def trainable_cpu():
    return OrderedDict((name,p.detach().float().cpu().contiguous().clone())
        for name,p in student.named_parameters() if p.requires_grad)
def save_best(step,q,reason):
    save_file(trainable_cpu(),str(BEST_FILE),metadata={
      "format":"pt","teacher_revision":PINNED_TEACHER_REVISION,
      "parameter_fingerprint":training_fingerprint,"step":str(step)})
    BEST_META.write_text(json.dumps(dict(
      step=step,quality=q,reason=reason,
      teacher_revision=PINNED_TEACHER_REVISION,
      parameter_fingerprint=training_fingerprint),indent=2),encoding="utf-8")
def restore_best():
    weights=load_file(str(BEST_FILE),device="cpu")
    assert set(weights)==set(TRAINABLE_PARAMETER_NAMES)
    with torch.no_grad():
        for name,p in student.named_parameters():
            if name in weights:
                assert p.shape==weights[name].shape,name
                p.copy_(weights[name].to(device=p.device,dtype=p.dtype))

def gold_gain_pp(q):
    return 100*(q["gold_accuracy"]-q["teacher_gold_accuracy"])
def valid_gold_candidate(q):
    return (q["teacher_agreement"]>=MIN_VAL_TEACHER_AGREEMENT and
            q["mean_js"]<=MAX_TEST_MEAN_JS)
def selection_score(q):
    return (int(valid_gold_candidate(q)),
            q["gold_accuracy"],
            -q["mean_js"],
            q["teacher_agreement"])

# Explicit hard gold and gold-probability vector must use marker order.
# Print conflicting labels as a data-quality diagnostic; these are NOT fixed
# by incorrectly modifying gold metadata or test labels.
gold_train_label_disagreement=sum(
    int(int(np.argmax(x["gold_probs"]))!=x["gold_index"]) for x in train_items)
print("Train gold labels vs max gold-probability disagreements:",
      gold_train_label_disagreement,"/",len(train_items))
def make_supervised_batch(step):
    offset=((step-1)*TRAIN_B)%len(train_items)
    examples=[train_items[(offset+j)%len(train_items)] for j in range(TRAIN_B)]
    b=direct_batch(examples)
    assert all(ex["gold_index"]<len(ex["markers"]) for ex in examples)
    assert bool((b["marker_pos"][b["marker_mask"]]<TRAIN_MAX_TOKENS).all())
    for k in ("input_ids","attention_mask"):
        b[k]=b[k][:,:TRAIN_MAX_TOKENS]
    hard=torch.tensor([x["gold_index"] for x in examples],
                      dtype=torch.long,device=DEVICE)
    soft=torch.zeros(b["marker_mask"].shape,dtype=torch.float32,device=DEVICE)
    for i,x in enumerate(examples):
        vals=torch.tensor(x["gold_probs"],dtype=torch.float32,device=DEVICE)
        assert len(vals)==len(x["markers"])
        soft[i,:len(vals)]=vals/vals.sum().clamp_min(1e-8)
    return b,hard,soft

best_quality=quality(student,gold_val_items,GOLD_VAL_REF)
initial_gold_val=best_quality
best_step=0
early_stop_reason=None
history=[]
save_best(0,best_quality,"initial_checkpoint")
optimizer=torch.optim.AdamW(groups,weight_decay=0.01)
scaler=torch.amp.GradScaler("cuda",enabled=amp_dtype==torch.float16)

for step in range(1,MAX_TRAIN_STEPS+1):
    b,hard,soft=make_supervised_batch(step)
    with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
        teacher_logits,_=teacher(**b)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda",dtype=amp_dtype):
        student_logits,_=student(**b)
    # Float32 softmax/loss and strict exclusion of padded choice markers.
    valid=b["marker_mask"].bool()
    student_scores=student_logits.float().masked_fill(~valid,-1e9)
    teacher_scores=teacher_logits.float().masked_fill(~valid,-1e9)
    logp_gold=F.log_softmax(student_scores,dim=-1)
    # Native F.cross_entropy(label_smoothing=...) also smooths INVALID
    # padded options. Their -1e9 logits yield an enormous false loss.
    # Smooth over the valid action choices ONLY.
    gold_one_hot=F.one_hot(hard,num_classes=student_scores.shape[-1]).float()
    valid_uniform=valid.float()/valid.sum(-1,keepdim=True).clamp_min(1)
    gold_targets=((1-LABEL_SMOOTHING)*gold_one_hot+
                  LABEL_SMOOTHING*valid_uniform)
    supervised=-(gold_targets*logp_gold).sum(-1).mean()
    gold_soft=-(soft*logp_gold).sum(-1).mean()
    kl=F.kl_div(F.log_softmax(student_scores/TEMPERATURE,dim=-1),
        F.softmax(teacher_scores/TEMPERATURE,dim=-1),
        reduction="batchmean")*TEMPERATURE**2
    loss=GOLD_CE_WEIGHT*supervised+SOFT_GOLD_WEIGHT*gold_soft+TEACHER_KL_WEIGHT*kl
    if not bool(torch.isfinite(loss)):
        raise RuntimeError(f"Non-finite gold training loss at step {step}")
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(trainable,1.0)
    scaler.step(optimizer)
    scaler.update()
    if step%CHECK_EVERY_STEPS==0 or step==MAX_TRAIN_STEPS:
        q=quality(student,gold_val_items,GOLD_VAL_REF)
        improved=selection_score(q)>selection_score(best_quality)
        if improved:
            best_quality=q
            best_step=step
            save_best(step,q,"best_gold_validation")
        rec=dict(step=step,train_loss=float(loss.detach().cpu()),
            gold_ce=float(supervised.detach().cpu()),
            gold_soft=float(gold_soft.detach().cpu()),
            kl_teacher=float(kl.detach().cpu()),
            quality=q,gold_gain_pp=gold_gain_pp(q),
            selected=improved)
        history.append(rec)
        TRAIN_LOG.write_text(json.dumps(history,indent=2),encoding="utf-8")
        print(f"STEP {step}: new-VAL gold={q['gold_accuracy']:.2%}, "
              f"teacher gold={q['teacher_gold_accuracy']:.2%}, "
              f"gain={gold_gain_pp(q):+.2f}pp, teacher agree="
              f"{q['teacher_agreement']:.2%}, JS={q['mean_js']:.5f}, "
              f"selected={improved}")
        if (valid_gold_candidate(q) and gold_gain_pp(q)>=EARLY_STOP_GOLD_GAIN_PP
                and step>=2*CHECK_EVERY_STEPS):
            # Reconfirm identical validation scores without an optimizer step.
            confirm=quality(student,gold_val_items,GOLD_VAL_REF)
            if (valid_gold_candidate(confirm) and
                 gold_gain_pp(confirm)>=EARLY_STOP_GOLD_GAIN_PP):
                best_step=step
                best_quality=q
                save_best(step,q,"gold_improvement_early_stop")
                early_stop_reason="validated_gold_gain_target_reached"
                print("EARLY STOP: new gold-validation improvement reached.")
                break

restore_best()
student.eval().requires_grad_(False)
final_gold_val=quality(student,gold_val_items,GOLD_VAL_REF)
historical_probe=quality(student,probe_items,PROBE_REF)
assert abs(final_gold_val["gold_accuracy"]-best_quality["gold_accuracy"])<1e-9
summary=dict(selected_step=best_step,stop_reason=early_stop_reason,
  source_revision=PINNED_TEACHER_REVISION,
  gold_validation_initial=initial_gold_val,gold_validation_final=final_gold_val,
  gold_validation_gain_pp=gold_gain_pp(final_gold_val),
  historical_probe=historical_probe,
  label_probability_disagreements=gold_train_label_disagreement,
  trainable_parameter_count=sum(p.numel() for p in trainable),
  selected_adapters_trained=TRAIN_COMPRESSED_ADAPTERS,
  training_log=history)
(OUT/"gold_training_summary.json").write_text(json.dumps(summary,indent=2))
print("SELECTED checkpoint:",best_step)
print("FINAL gold-validation improvement:",gold_gain_pp(final_gold_val),"pp")


PASS: valid-option gold loss and gradient sanity check
Trainable GOLD-aware parameters: 1055745 adapters enabled: False
Train gold labels vs max gold-probability disagreements: 23 / 2500
STEP 20: new-VAL gold=83.33%, teacher gold=85.00%, gain=-1.67pp, teacher agree=97.78%, JS=0.00260, selected=True
STEP 40: new-VAL gold=83.33%, teacher gold=85.00%, gain=-1.67pp, teacher agree=97.78%, JS=0.00229, selected=True
STEP 60: new-VAL gold=83.33%, teacher gold=85.00%, gain=-1.67pp, teacher agree=97.78%, JS=0.00201, selected=True
STEP 80: new-VAL gold=83.33%, teacher gold=85.00%, gain=-1.67pp, teacher agree=97.78%, JS=0.00187, selected=True
STEP 100: new-VAL gold=83.33%, teacher gold=85.00%, gain=-1.67pp, teacher agree=97.78%, JS=0.00192, selected=False
STEP 120: new-VAL gold=83.33%, teacher gold=85.00%, gain=-1.67pp, teacher agree=97.78%, JS=0.00249, selected=False
STEP 140: new-VAL gold=83.33%, teacher gold=85.00%, gain=-1.67pp, teacher agree=97.78%, JS=0.00304, selected=False
STEP 160: new-VA

In [8]:
#@title 8. ONE distinct TEST-split AUDIT and paired GPU latency
# The test examples have NEVER been used to select ranks or checkpoints.
# Freeze all parameters before touching the test split here.
assert not any(p.requires_grad for p in student.parameters())
VAL_BENCH_BATCHES=BENCH_BATCHES
BENCH_BATCHES=[batch(gold_val_items,0,BATCH),batch(gold_val_items,11,BATCH)]
validation_speed=benchmark_paired(teacher,student,repeats=BENCH_REPEATS)
BENCH_BATCHES=[batch(audit_items,0,BATCH),batch(audit_items,11,BATCH)]
TEST_REF=get_reference(audit_items,AUDIT_DECISIONS)
assert {r["case"] for r in TEST_REF}.isdisjoint(
    {r["case"] for r in GOLD_VAL_REF})
test_quality=quality(student,audit_items,TEST_REF)
test_speed=benchmark_paired(teacher,student,repeats=max(15,BENCH_REPEATS))
BENCH_BATCHES=VAL_BENCH_BATCHES
test_gold_gain_pp=100*(test_quality["gold_accuracy"]-
    test_quality["teacher_gold_accuracy"])
print("FINAL TEST GOLD student:",test_quality["gold_accuracy"])
print("FINAL TEST GOLD teacher:",test_quality["teacher_gold_accuracy"])
print("TEST gold advantage:",test_gold_gain_pp,"percentage points")
print("TEST teacher agreement:",test_quality["teacher_agreement"])
print("Full model GPU speed ratio:",test_speed["median_gpu_speedup"],
      "minimum:",test_speed["min_gpu_speedup"])


FINAL TEST GOLD student: 0.755
FINAL TEST GOLD teacher: 0.765
TEST gold advantage: -1.0000000000000009 percentage points
TEST teacher agreement: 0.93
Full model GPU speed ratio: 1.020125387139798 minimum: 1.0150395577709237


In [9]:
#@title 9. Gold error breakdown (from frozen TEST predictions; no retraining)
# Report where teacher-only errors are corrected and newly introduced errors.
from collections import Counter
@torch.inference_mode()
def comparison_breakdown(items,ref):
    counters=Counter()
    qtypes={}
    for offset in range(0,len(ref),BATCH):
        selected=items[offset:offset+BATCH]
        probs=probs_for(student,direct_batch(selected))
        for j,item in enumerate(selected):
            p=probs[j]; truth=item["gold_index"]
            t=ref[offset+j]["teacher"];s=int(p.argmax())
            kind=str(item["qtype_name"])
            ct=qtypes.setdefault(kind,Counter())
            if t==truth and s==truth:
                label="both_correct"
            elif t!=truth and s==truth:
                label="teacher_wrong_student_correct"
            elif t==truth and s!=truth:
                label="teacher_correct_student_wrong"
            else:
                label="both_wrong"
            counters[label]+=1;ct[label]+=1
    return dict(overall=dict(counters),
                by_question_type={k:dict(v) for k,v in qtypes.items()})
test_breakdown=comparison_breakdown(audit_items,TEST_REF)
print(json.dumps(test_breakdown,indent=2))
(OUT/"test_error_breakdown.json").write_text(
    json.dumps(test_breakdown,indent=2))


{
  "overall": {
    "both_correct": 145,
    "both_wrong": 41,
    "teacher_correct_student_wrong": 8,
    "teacher_wrong_student_correct": 6
  },
  "by_question_type": {
    "choice": {
      "both_correct": 41,
      "both_wrong": 13,
      "teacher_wrong_student_correct": 2
    },
    "score": {
      "both_correct": 58,
      "teacher_correct_student_wrong": 6,
      "both_wrong": 16,
      "teacher_wrong_student_correct": 4
    },
    "noul": {
      "both_correct": 46,
      "both_wrong": 12,
      "teacher_correct_student_wrong": 2
    }
  }
}


557

In [10]:
#@title 10. Save COMPLETE offline FP32 model, verify strict reload and disk savings
# Independently recompute the fingerprint HERE as well.
# This lets users resume from a completed training session and rerun only
# this export cell, without re-running gold training or downloading weights.
spec_hash=hashlib.sha256(
    json.dumps(SELECTED_SPECS,sort_keys=True,separators=(",",":")).encode("utf-8")
).hexdigest()
assert len(spec_hash)==64
print("Exporting selected compression specification:",spec_hash)
(DEST/"encoder").mkdir(parents=True,exist_ok=True)
(DEST/"tokenizer").mkdir(parents=True,exist_ok=True)
student.encoder.config.save_pretrained(str(DEST/"encoder"))
tokenizer.save_pretrained(str(DEST/"tokenizer"))
(DEST/"modeling_laya_v50.py").write_text(V50_RUNTIME_CODE,encoding="utf-8")
metadata=dict(format="laya-v50-all28-bruteforce-v1",
 model_type="laya-v52-gold-supervised-hybrid-svd",
 head_context_budget=64,
 head_layers=int(source_config["head_layers"]),
 source_model=TEACHER_REPO,source_revision=PINNED_TEACHER_REVISION,
 compressed_projections=SELECTED_SPECS,
 source_model_layers=28,selected_training_step=best_step,
 early_stop_reason=early_stop_reason,
 selected_spec_sha256=spec_hash,
 output_scope="typed-decision option scores only, no auxiliary action logits")
(DEST/"config.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
state=student.state_dict()
assert not any("._shared." in name for name in state)
cpu_tensors=OrderedDict(
    (name,value.detach().cpu().contiguous().clone())
    for name,value in state.items())
weights=DEST/"model.safetensors"
save_file(cpu_tensors,str(weights),metadata={
    "format":"pt","source_revision":PINNED_TEACHER_REVISION,
    "architecture":"laya-v52-gold-supervised-v50-loader"})
del cpu_tensors,state
gc.collect()
original_bytes=(parent_dir/"model.safetensors").stat().st_size
model_bytes=weights.stat().st_size
actual_saved_bytes=original_bytes-model_bytes
assert actual_saved_bytes>0,(original_bytes,model_bytes)
print("Actual FULL model checkpoint:",model_bytes,"bytes")
print("Reference checkpoint:",original_bytes,"bytes")
print("Actual saved bytes:",actual_saved_bytes)

reloaded,reloaded_tokenizer=v50.load_v50_model(DEST,device="cuda")
smoke=batch(probe_items,0,BATCH)
with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
    a,_=student(**smoke)
    b,_=reloaded(**smoke)
max_diff=float((a.float()-b.float()).abs().max())
choices_identical=bool(torch.equal(a.argmax(-1),b.argmax(-1)))
logits_close=bool(torch.allclose(a.float(),b.float(),rtol=2e-3,atol=2e-3))
assert choices_identical and logits_close,(max_diff,choices_identical,logits_close)
del reloaded
gc.collect()
torch.cuda.empty_cache()
print("STANDALONE LOCAL RELOAD: PASS, max logit difference:",max_diff)


Exporting selected compression specification: 5cb1804e2e2bea0180c19f3179f3075c9be467e49cc3ac99257f3318f4c6ac6e
Actual FULL model checkpoint: 1553546100 bytes
Reference checkpoint: 1684141004 bytes
Actual saved bytes: 130594904
STANDALONE LOCAL RELOAD: PASS, max logit difference: 0.0


In [11]:
#@title 11. Research release gates, model card and conditional HF publication
val_gain=gold_gain_pp(final_gold_val)
test_gain=test_gold_gain_pp
speed_nonreg=(test_speed["min_gpu_speedup"]>=MIN_GPU_SPEED_RATIO and
              test_speed["min_wall_speedup"]>=MIN_WALL_SPEED_RATIO)
val_ok=(val_gain>=MIN_VALID_GOLD_GAIN_PP and
        final_gold_val["teacher_agreement"]>=MIN_VAL_TEACHER_AGREEMENT and
        final_gold_val["mean_js"]<=MAX_TEST_MEAN_JS)
test_ok=(test_gain>=MIN_TEST_GOLD_GAIN_PP and
         test_quality["teacher_agreement"]>=MIN_TEST_TEACHER_AGREEMENT and
         test_quality["mean_js"]<=MAX_TEST_MEAN_JS)
gates=dict(
 fresh_gold_validation_gain=val_ok,
 independent_gold_test_gain=test_ok,
 checkpoint_saving_at_least_20MB=actual_saved_bytes>=MIN_SAVED_BYTES,
 offline_reload_parity=choices_identical and logits_close,
 speed_nonregression=speed_nonreg)
gates["all_pass"]=bool(
    gates["fresh_gold_validation_gain"] and gates["independent_gold_test_gain"]
    and gates["checkpoint_saving_at_least_20MB"] and gates["offline_reload_parity"]
    and (speed_nonreg or not REQUIRE_SPEED_NONREGRESSION))
report=dict(version="V5.2",experiment="gold-supervised teacher correction",
 source_repo=TEACHER_REPO,source_revision=PINNED_TEACHER_REVISION,
 compressed_projection_count=len(SELECTED_SPECS),
 compressed_projections=SELECTED_SPECS,
 selected_step=best_step,early_stop_reason=early_stop_reason,
 gold_training_loss_weights=dict(hard=GOLD_CE_WEIGHT,
   soft=SOFT_GOLD_WEIGHT,teacher_kl=TEACHER_KL_WEIGHT),
 train_selected_adapters=TRAIN_COMPRESSED_ADAPTERS,
 gold_train_label_probability_disagreements=gold_train_label_disagreement,
 gold_validation=final_gold_val,gold_validation_gain_pp=val_gain,
 old_dev_probe_diagnostic=historical_probe,
 independent_test=test_quality,independent_test_gain_pp=test_gain,
 independent_test_error_breakdown=test_breakdown,
 test_latency=test_speed,validation_latency=validation_speed,
 native_unique_parameters=native_unique,
 new_unique_parameters=unique_student,
 original_checkpoint_bytes=original_bytes,new_checkpoint_bytes=model_bytes,
 measured_storage_savings_bytes=actual_saved_bytes,
 reload_max_abs_difference=max_diff,
 speed_nonregression_required=REQUIRE_SPEED_NONREGRESSION,gates=gates,
 limitation="Gold validation was used repeatedly for checkpoint selection; only the separate dataset TEST split is an untouched audit. The test is small; positive score differences may not be statistically significant. Not all gold labels are necessarily unambiguous.")
(DEST/"v52_report.json").write_text(json.dumps(report,indent=2))
card=f"""---
library_name: pytorch
base_model: {TEACHER_REPO}
tags:
- modernbert
- gold-supervised
- svd
- model-compression
- standalone
---

# Laya V5.2 — Gold-Supervised Compressed ModernBERT

V5.1 matched 100% of the 120 repeatedly used teacher PROBE decisions
but teacher and student gold accuracy were both 87.5%. V5.2 uses actual
gold labels (with teacher-KL regularization), aiming to **correct
pretrained teacher errors** instead of reproducing them.

All 28 original ModernBERT encoder layers are retained; 20 attention/FFN
projections have compact SVD/pair-delta factors. The scorer and question
type embeddings received gold supervision.
Optional adapters training enabled: **{TRAIN_COMPRESSED_ADAPTERS}**.

## Independently measured TEST split (not development PROBE)

- Teacher gold accuracy: **{test_quality["teacher_gold_accuracy"]:.3%}**
- V5.2 gold accuracy: **{test_quality["gold_accuracy"]:.3%}**
- Gold advantage: **{test_gain:+.2f} percentage points**
- Teacher decision agreement: **{test_quality["teacher_agreement"]:.3%}**
- Gold validation improvement: **{val_gain:+.2f} percentage points**
- Source revision: `{PINNED_TEACHER_REVISION}`
- Selected optimizer step: **{best_step}**
- FP32 model weight bytes: **{model_bytes:,}**
- Reference FP32 weight bytes: **{original_bytes:,}**
- Measured checkpoint storage reduction: **{actual_saved_bytes:,} bytes**
- Median GPU speed ratio (V4.6/V5.2): **{test_speed["median_gpu_speedup"]:.3f}x**
- All release acceptance gates passed: **{gates["all_pass"]}**

Gold improvement is not the same as teacher agreement and does not
prove generalization beyond this dataset. If speed ratio is below 1,
the model is smaller but NOT faster.

## Fully offline inference after one download

```python
from modeling_laya_v50 import load_v50_model
model,tokenizer=load_v50_model("/path/to/model",device="cuda")
answers=model.predict(tokenizer,dict(road_direction="left",speed=7),
    dict(steering=dict(type="choice",instructions="Choose direction",
    criteria=dict(left="left",straight="straight",right="right"))))
print(answers)
```

Contains complete FP32 weights, tokenizer, ModernBERT config,
inference runtime and measured reports. No parent download needed.
"""
(DEST/"README.md").write_text(card,encoding="utf-8")
(DEST/"requirements.txt").write_text(
 "torch>=2.4\ntransformers==4.57.6\nsafetensors>=0.4\nhuggingface_hub>=0.25,<1\n")
(DEST/"provenance.json").write_text(json.dumps(dict(
 source_model=TEACHER_REPO,source_revision=PINNED_TEACHER_REVISION,
 selected_specs=SELECTED_SPECS,selected_spec_sha256=spec_hash,
 selected_step=best_step,supervised_gold=True),indent=2))
def folder_bytes(root):
    return sum(f.stat().st_size for f in Path(root).rglob("*") if f.is_file())
report["standalone_dir_bytes"]=folder_bytes(DEST)
(DEST/"v52_report.json").write_text(json.dumps(report,indent=2))
print("GATES:",json.dumps(gates,indent=2))
print("REPORT:",DEST/"v52_report.json")
print("STANDALONE CHECKPOINT:",weights)
HF_UPLOAD_STATUS = "not_attempted"
HF_UPLOAD_ERROR = None
import traceback
try:
    if not gates["all_pass"]:
        print("NOT VALIDATED: keep local experimental checkpoint; HF publication BLOCKED.")
    elif PUBLISH_TO_HF:
        notebook_login()
        uploader=HfApi()
        account=uploader.whoami()["name"]
        assert account=="vtava",f"Expected vtava write access, got {account!r}"
        assert STUDENT_REPO=="vtava/Laya-v52-gold-aware-standalone"
        if uploader.repo_exists(STUDENT_REPO,repo_type="model"):
            existing=set(uploader.list_repo_files(STUDENT_REPO,repo_type="model"))
            if existing-{".gitattributes",".gitignore"} and not ALLOW_UPDATE_EXISTING:
                raise RuntimeError("Refusing to overwrite a populated V5.2 repository.")
        else:
            uploader.create_repo(STUDENT_REPO,repo_type="model",
                                 private=PRIVATE_REPO,exist_ok=False)
        uploader.upload_large_folder(repo_id=STUDENT_REPO,repo_type="model",
                                     folder_path=str(DEST))
        remote=set(uploader.list_repo_files(STUDENT_REPO,repo_type="model"))
        required={"model.safetensors","modeling_laya_v50.py","encoder/config.json",
                  "config.json","README.md","v52_report.json","provenance.json"}
        assert required.issubset(remote),(required-remote)
        print("HF MODEL PUBLISHED:",f"https://huggingface.co/{STUDENT_REPO}")
    else:
        print("HF publishing disabled; model kept locally.")

    HF_UPLOAD_STATUS = ("blocked_by_gates" if not gates["all_pass"] else
                        "published" if PUBLISH_TO_HF else "disabled")
except Exception as exc:
    HF_UPLOAD_STATUS = "upload_error"
    HF_UPLOAD_ERROR = f"{type(exc).__name__}: {exc}"
    (OUT/"huggingface_upload_error.txt").write_text(traceback.format_exc(),encoding="utf-8")
    print("HF publication error captured; continuing to ZIP download:",HF_UPLOAD_ERROR)
(OUT/"huggingface_publication_status.json").write_text(json.dumps({
    "status":HF_UPLOAD_STATUS,"error":HF_UPLOAD_ERROR,
    "repo":STUDENT_REPO,"all_gates_pass":gates["all_pass"],
    "gates":gates},indent=2),encoding="utf-8")
print("Archive cell will run regardless of Hugging Face release status.")


GATES: {
  "fresh_gold_validation_gain": false,
  "independent_gold_test_gain": false,
  "checkpoint_saving_at_least_20MB": true,
  "offline_reload_parity": true,
  "speed_nonregression": true,
  "all_pass": false
}
REPORT: /content/Laya_V52_GoldAware/standalone_model/v52_report.json
STANDALONE CHECKPOINT: /content/Laya_V52_GoldAware/standalone_model/model.safetensors
NOT VALIDATED: keep local experimental checkpoint; HF publication BLOCKED.


In [ ]:
#@title 12. ZIP ALL V5.2 results, checkpoints, full model and AUTO-DOWNLOAD
# This cell is deliberately independent of the HF publication gates.
# It is safe to run again without retraining or uploading to HF.
import json,hashlib,zipfile
from pathlib import Path
from datetime import datetime,timezone

if "OUT" not in globals():
    OUT=Path("/content/Laya_V52_GoldAware")
OUT=Path(OUT).resolve()
assert OUT.is_dir(),f"Run output directory missing: {OUT}"
MODEL_DIR=OUT/"standalone_model"
ZIP_PATH=OUT.parent/"Laya_V52_All_Results.zip"

# Explicit summary for reviewers; do not claim success when release gates fail.
summary={
    "experiment":"Laya V5.2 gold-aware supervised compressed model",
    "generated_utc":datetime.now(timezone.utc).isoformat(),
    "notebook":"Laya_V52_Gold_Aware_Supervised_Colab.ipynb",
    "results_dir":str(OUT),
    "model_directory_in_zip":"Laya_V52_GoldAware/standalone_model/",
}
if "gates" in globals():
    summary["release_gates"]=gates
if "HF_UPLOAD_STATUS" in globals():
    summary["hugging_face_status"]=HF_UPLOAD_STATUS
if "HF_UPLOAD_ERROR" in globals():
    summary["hugging_face_error"]=HF_UPLOAD_ERROR
if "best_step" in globals():
    summary["selected_training_step"]=int(best_step)
if "final_gold_val" in globals():
    summary["validation"]=final_gold_val
if "test_quality" in globals():
    summary["independent_test"]=test_quality
if "test_gold_gain_pp" in globals():
    summary["test_gold_gain_pp"]=float(test_gold_gain_pp)
(OUT/"all_results_summary.json").write_text(json.dumps(
    summary,indent=2,default=str),encoding="utf-8")

# Collect every generated artifact under OUT, including all saved logs,
# the best trainable-parameter checkpoint, the entire standalone encoder,
# tokenizer, model.safetensors, model card, provenance and final report.
# Do NOT include the downloaded teacher or Hugging Face cache: they are not
# part of the new standalone artifact. The ZIP lives outside OUT.
MANIFEST_PATH=OUT/"archive_manifest.json"
files_to_zip=sorted(p for p in OUT.rglob("*")
                    if p.is_file() and not p.is_symlink()
                    and p.resolve()!=MANIFEST_PATH.resolve()
                    and p.resolve()!=ZIP_PATH.resolve())
assert files_to_zip,"No result files found"
names={str((Path(OUT.name)/p.relative_to(OUT)).as_posix())
       for p in files_to_zip}
# An incomplete run is still downloadable; record missing outputs explicitly.
required_paths={
    "standalone_model/model.safetensors",
    "standalone_model/config.json",
    "standalone_model/modeling_laya_v50.py",
    "standalone_model/v52_report.json",
    "gold_training_progress.json",
    "gold_training_summary.json",
    "test_error_breakdown.json",
    "best_gold_trainable.safetensors",
}
present={p.relative_to(OUT).as_posix() for p in files_to_zip}
missing=sorted(required_paths-present)
if missing:
    print("WARNING: run has incomplete outputs; archiving everything available.")
    print("Missing expected output files:",missing)

def sha256_file(source):
    digest=hashlib.sha256()
    with open(source,"rb") as reader:
        for chunk in iter(lambda:reader.read(8*1024*1024),b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest={
    "archive_format":"laya-v52-results-v1",
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "contains_complete_standalone_model":len(missing)==0,
    "missing_expected_files":missing,
    "files":[
        {"path":(Path(OUT.name)/p.relative_to(OUT)).as_posix(),
         "bytes":p.stat().st_size,"sha256":sha256_file(p)}
        for p in files_to_zip
    ]
}
MANIFEST_PATH.write_text(json.dumps(manifest,indent=2),encoding="utf-8")
all_files=sorted(files_to_zip+[MANIFEST_PATH])
print("Packing",len(all_files),"files; source bytes:",
      sum(p.stat().st_size for p in all_files))
print("The full FP32 model may make the ZIP exceed 1 GB.")

# Streaming ZIP64, with no model bytes loaded into RAM.
# Safetensors are stored rather than redundantly compressed for speed.
with zipfile.ZipFile(ZIP_PATH,mode="w",allowZip64=True,
                     compression=zipfile.ZIP_DEFLATED,compresslevel=1) as out_zip:
    for p in all_files:
        arcname=(Path(OUT.name)/p.relative_to(OUT)).as_posix()
        method=(zipfile.ZIP_STORED if p.suffix in
                {".safetensors",".pt",".pth",".zip",".gz"} else
                zipfile.ZIP_DEFLATED)
        out_zip.write(p,arcname=arcname,compress_type=method)
# Validate the ZIP entries and byte-size metadata before launching download.
with zipfile.ZipFile(ZIP_PATH,mode="r") as check:
    expected={(Path(OUT.name)/p.relative_to(OUT)).as_posix()
              for p in all_files}
    actual=set(check.namelist())
    assert actual==expected,("ZIP entry mismatch",sorted(expected-actual),
                             sorted(actual-expected))
    assert all(check.getinfo(name).file_size==
               (OUT/Path(name).relative_to(OUT.name)).stat().st_size
               for name in actual)
    assert check.getinfo(f"{OUT.name}/archive_manifest.json").file_size>0

print("PASS: Full results ZIP created and entries verified")
print("ZIP:",ZIP_PATH,"bytes:",ZIP_PATH.stat().st_size)
print("Result files:",len(all_files),"missing:",len(missing))

# Auto-download: no extra user action is needed when running in Colab.
# If a browser blocks a very large file, the ZIP remains in /content,
# and a direct link is also displayed.
try:
    from IPython.display import display,FileLink
    display(FileLink(str(ZIP_PATH)))
except ImportError:
    pass
try:
    from google.colab import files as colab_files
    colab_files.download(str(ZIP_PATH))
    print("Download initiated:",ZIP_PATH.name)
except (ImportError,Exception) as download_error:
    print("Automatic browser download unavailable or blocked:",
          repr(download_error))
    print("The completed ZIP remains available at:",ZIP_PATH)
